In [2]:
%pip install pyodbc

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import pyodbc

server = 'aviation-sql-0223.database.windows.net'
database = 'aviation-fault-db'
username = 'YOUR_USERNAME_HERE'
password = 'YOUR_PASSWORD_HERE'

driver = '{ODBC Driver 17 for SQL Server}'
connection_string = f'DRIVER={driver};SERVER={server};PORT=1433;DATABASE={database};UID={username};PWD={password}'

In [23]:
print("Connecting to Azure SQL...")
try:
    conn = pyodbc.connect(connection_string)
    print("Connection Successful!")
except Exception as e:
    print("Connection failed. Error:")
    print(e)

Connecting to Azure SQL...
Connection Successful!


In [6]:
query = "SELECT Discrepancy, JASCCode FROM dbo.CleanAviationMaintenanceData"
df = pd.read_sql(query, conn)
print(f"Success! Loaded {len(df)} rows into the lab.")

/tmp/ipykernel_4400/2690786709.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [7]:
df.head()

,Discrepancy,JASCCode
0,FWD E/E COMP INSIDE COMP LH SIDE STA 457-475 F...,5311
1,(A204 DOT 73) FORWARD CARGO STA 500 TO 500A; S...,5320
2,FWD FLASHLIGHT CHARGING INDICATOR INOPERATIVE....,2560
3,FWD CARGO FLOOR PANEL 120CF HAS CORROSION AT F...,5321
4,LH WS220 LOWER WING SKIN HAS CHAFFING COMMON T...,5730


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Define Features (X) and Target (y)
X = df['Discrepancy']
y = df['JASCCode']

# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Data split successful! Training rows: {len(X_train)} | Testing rows: {len(X_test)}")

Data split successful! Training rows: 580 | Testing rows: 146


In [9]:
print("Vectorizing the mechanic logs...")

# Initialize the TF-IDF translator, ignoring common filler words like 'the' or 'and'
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)

# Learn the vocabulary from the training set and transform the text into numbers
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vectorization complete.")

Vectorizing the mechanic logs...
Vectorization complete.


In [10]:
print("Training the classification model...")

# Initialize and train the AI
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Grade the AI on the test set
y_pred = model.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)

print("-" * 30)
print(f"Model Training Complete!")
print(f"AI Accuracy Score: {accuracy * 100:.2f}%")

Training the classification model...
------------------------------
Model Training Complete!
AI Accuracy Score: 45.89%


In [11]:
# 1. Let's check how hard the test actually is
num_classes = df['JASCCode'].nunique()
print(f"The AI is trying to guess between {num_classes} different fault codes!")
print("-" * 30)

# 2. Upgrade the Vectorizer: Capture 2-word phrases (N-grams) and ignore ultra-rare typos
print("Re-vectorizing using Bigrams...")
upgraded_vectorizer = TfidfVectorizer(stop_words='english', max_features=2500, ngram_range=(1, 2), min_df=2)

X_train_vec_up = upgraded_vectorizer.fit_transform(X_train)
X_test_vec_up = upgraded_vectorizer.transform(X_test)

# 3. Upgrade the Model: Let it learn closer to the training data (C=10)
print("Training the upgraded model...")
upgraded_model = LogisticRegression(max_iter=2000, C=10)
upgraded_model.fit(X_train_vec_up, y_train)

# 4. Grade the new AI
y_pred_up = upgraded_model.predict(X_test_vec_up)
new_accuracy = accuracy_score(y_test, y_pred_up)

print("-" * 30)
print(f"Upgraded AI Accuracy Score: {new_accuracy * 100:.2f}%")

The AI is trying to guess between 119 different fault codes!
------------------------------
Re-vectorizing using Bigrams...
Training the upgraded model...
------------------------------
Upgraded AI Accuracy Score: 67.12%


In [12]:
import numpy as np

def predict_mechanic_log(messy_text):
    print(f"Mechanic Log: '{messy_text}'\n" + "-"*40)
    
    # 1. Translate the new sentence into math
    vectorized_text = upgraded_vectorizer.transform([messy_text])
    
    # 2. Predict the JASC Code
    predicted_code = upgraded_model.predict(vectorized_text)[0]
    
    # 3. Calculate the Confidence Score (Probability)
    probabilities = upgraded_model.predict_proba(vectorized_text)[0]
    confidence_score = max(probabilities) * 100
    
    print(f" AI Prediction: JASC Code {predicted_code}")
    print(f" Confidence Score: {confidence_score:.2f}%")
    
    if confidence_score < 60:
        print(" Warning: Low confidence. Flagging for Human-in-the-Loop review.")

# Let's test it with a fake, messy log! 
test_log = "FOUND MAJOR CORROSION AND HUGE CRACKS ON THE LEFT LANDING GEAR DOOR IN THE FWD CARGO AREA."

predict_mechanic_log(test_log)

Mechanic Log: 'FOUND MAJOR CORROSION AND HUGE CRACKS ON THE LEFT LANDING GEAR DOOR IN THE FWD CARGO AREA.'
----------------------------------------
 AI Prediction: JASC Code 5320
 Confidence Score: 12.18%


In [24]:
import pandas as pd

print("Scoring the entire dataset with the AI...")

# 1. Pull the data manually using pyodbc to bypass Pandas strictness
query = "SELECT OperatorControlNumber, Discrepancy FROM dbo.CleanAviationMaintenanceData"
cursor = conn.cursor()
cursor.execute(query)

# Grab all the rows and column names, then build the DataFrame manually
rows = cursor.fetchall()
columns = [column[0] for column in cursor.description]
full_df = pd.DataFrame.from_records(rows, columns=columns)

# 2. Vectorize all the text
X_all_vec = upgraded_vectorizer.transform(full_df['Discrepancy'])

# 3. Make predictions and calculate confidence scores
full_df['Predicted_JASCCode'] = upgraded_model.predict(X_all_vec)
probabilities = upgraded_model.predict_proba(X_all_vec)
full_df['Confidence_Score'] = probabilities.max(axis=1) * 100

# 4. Write the results back to Azure SQL
print("Pushing predictions back to Azure SQL database...")

# Create a new table to hold the AI results
cursor.execute("""
    IF OBJECT_ID('dbo.AIPredictions', 'U') IS NOT NULL DROP TABLE dbo.AIPredictions;
    CREATE TABLE dbo.AIPredictions (
        OperatorControlNumber NVARCHAR(255),
        Predicted_JASCCode INT,
        Confidence_Score FLOAT
    )
""")

# Insert the rows one by one
for index, row in full_df.iterrows():
    cursor.execute("""
        INSERT INTO dbo.AIPredictions (OperatorControlNumber, Predicted_JASCCode, Confidence_Score)
        VALUES (?, ?, ?)
    """, row['OperatorControlNumber'], row['Predicted_JASCCode'], row['Confidence_Score'])

conn.commit()
print(f"Success! {len(full_df)} AI predictions safely stored in Azure SQL.")

Scoring the entire dataset with the AI...
Pushing predictions back to Azure SQL database...
Success! 726 AI predictions safely stored in Azure SQL.
